# Agentic System for Real-Time IT Incident Response
### LangChain / LangGraph | Agentic Pipeline
 **Key Conceptual Shift**

- Multi-agent: we define the workflow, agents execute it
- Agentic: the LLM defines the workflow at runtime, agents + tools execute it

Comparing with the `it_incident_multi-agent` notebook, modified cells are marked with *.

### Agentic AI
The system itself decides what to do next based on the situation. It can loop, skip steps, call tools, ask for clarification, or take autonomous actions. The key properties that make something "agentic":

| Property      |  Multi-Agent           | Agentic AI                          |
|---------------|------------------------|-------------------------------------|
| Routing       | Fixed sequence         | LLM decides next step               |
| Loops         | None                   | Can retry or re-evaluate            |
| Tool use      | None                   | Calls APIs, runs scripts            |
| Memory        | State object only      | Persists context across runs        |
| Autonomy      | Zero                   | Can act without being asked         |
| Clarification | Manual intervention    | Asks for clarification autonomously |

**Example**: a LOW severity alert goes through the full Analyzer + Fixer pipeline regardless. An agentic system would reason "_this is LOW severity, skip deep analysis, just log it_". If the root cause is unclear, loop back and query more data before proceeding.

## Cell 1 | Install Dependencies

In [0]:
%pip install langgraph langchain databricks-langchain -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

## Cell 2 | LLM Client (Databricks-hosted model)

In [0]:

from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",  # no endpoint setup
    temperature=0,
)

## * Cell 3 | Shared State Schema
Replace the `TypedDict` state with a message-based state.

In [0]:
from typing import TypedDict, List, Annotated
import operator

class AgentState(TypedDict):
    raw_alert:  str
    messages:   Annotated[List, operator.add]  # append-only log
    result:     dict                            # final structured output

## * Cell 4 | Agent Nodes
Replace agent functions with tools the LLM can choose from.
The core change is replacing the fixed graph with a ReAct-style agent that has tools and decides its own flow. 

In [0]:
from langchain_core.tools import tool

@tool
def collect_alert(raw_alert: str) -> str:
    """Parse and normalise a raw IT monitoring alert."""
    response = llm.invoke(
        f"Parse this alert and extract: source system, timestamp, "
        f"affected service, error type, impact scope.\n\nAlert: {raw_alert}"
    )
    return response.content

@tool
def screen_alert(alert: str) -> str:
    """Assign severity (LOW/MEDIUM/HIGH/CRITICAL) and urgency to an alert."""
    response = llm.invoke(
        f"Reply in JSON only: {{\"severity\": \"LOW|MEDIUM|HIGH|CRITICAL\", "
        f"\"urgency\": \"ROUTINE|URGENT|IMMEDIATE\"}}.\n\nAlert: {alert}"
    )
    return response.content

@tool
def analyze_alert(alert: str, severity: str) -> str:
    """Identify the root cause. Only call this for HIGH or CRITICAL severity."""
    response = llm.invoke(
        f"Identify the most likely root cause.\n\nAlert: {alert}\nSeverity: {severity}"
    )
    return response.content

@tool
def fix_incident(root_cause: str, severity: str) -> str:
    """Generate step-by-step remediation steps."""
    response = llm.invoke(
        f"Provide numbered remediation steps.\n\nRoot cause: {root_cause}\nSeverity: {severity}"
    )
    return response.content

@tool
def escalate(alert: str, reason: str) -> str:
    """Escalate to a human engineer when the agent cannot resolve the incident."""
    # In production: call PagerDuty / OpsGenie API here
    return f"ESCALATED: {reason} | Alert: {alert[:100]}"

tools = [collect_alert, screen_alert, analyze_alert, fix_incident, escalate]

## * Cell 5 | Build & Compile the Graph
 Replace the fixed graph with a ReAct agent loop.

In [0]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are an autonomous IT incident management agent.

When given an alert, you MUST:
1. Always call collect_alert first to normalize it
2. Always call screen_alert to determine severity and urgency
3. If severity is HIGH or CRITICAL → call analyze_alert, then fix_incident
4. If severity is LOW or MEDIUM → skip deep analysis, just summarise the finding
5. If you cannot determine the root cause after analysis → call escalate
6. Never fabricate information — only use what tools return

Be concise. Make decisions based on tool outputs, not assumptions.
"""

app = create_react_agent(
    llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

/home/spark-a494d756-df57-45b0-97dc-50/.ipykernel/4630/command-3419476384926890-3612639495:16: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  app = create_react_agent(


## * Cell 6 | Run the Pipeline
Invoke using message format.

**Simulate a monitoring tool pushing an alert (run anytime)**

In [0]:
from datetime import datetime

new_alert = spark.createDataFrame([
    ("CPU spike to 98% on prod-db-01 at 14:32 UTC. 500 errors on API gateway.", datetime.utcnow(), False),
    ("Disk usage at 95% on prod-storage-02. Write latency > 800ms.",             datetime.utcnow(), False),
], ["raw_alert", "created_at", "processed"])

new_alert.write.mode("append").saveAsTable("main.it_ops.alert_queue")
print("Alerts queued. Run process.")

Alerts queued. Run process.


`process_pending_alerts()` becomes "agentic-aware" through three key shifts:

- **Conversational Input**: Alerts are wrapped as a `HumanMessage` rather than a structured dictionary, initiating a reasoning session instead of a fixed data pipeline.
- **Chain-based Output**: Instead of returning discrete fields, the system produces a message chain; the final resolution is extracted from the last message in that sequence.
- **Comprehensive Tracing**: The entire interaction - every tool call, observation, and thought - is logged. This makes the agent’s logic fully auditable, showing how it reached a conclusion rather than just the final result

In [0]:
from langchain_core.messages import HumanMessage

def process_pending_alerts():
    from pyspark.sql.functions import col

    pending = spark.table("main.it_ops.alert_queue") \
                   .filter(col("processed") == False)
    rows = pending.collect()

    if not rows:
        print("No pending alerts.")
        return

    for row in rows:
        print(f"\nProcessing: {row.raw_alert[:60]}...")

        result = app.invoke({
            "messages": [HumanMessage(content=f"Handle this IT alert: {row.raw_alert}")]
        })

        # The agent's final message contains its reasoning + conclusion
        final = result["messages"][-1].content
        print(f"Agent conclusion:\n{final}")

        # Log the full reasoning trace to Delta
        trace = "\n".join([m.content for m in result["messages"] if hasattr(m, "content")])
        spark.createDataFrame(
            [(row.raw_alert[:500], final[:1000], trace[:5000])],
            ["alert", "conclusion", "trace"]
        ).write.mode("append").option("mergeSchema", "true").saveAsTable("main.it_ops.incident_log")

        spark.sql(f"""
            UPDATE main.it_ops.alert_queue SET processed = TRUE
            WHERE raw_alert = '{row.raw_alert.replace("'", "''")}'
        """)



In [0]:
process_pending_alerts()


Processing: CPU spike to 98% on prod-db-01 at 14:32 UTC. 500 errors on A...
Agent conclusion:
The incident has been analyzed and remediation steps have been provided to address the root cause of inefficient or long-running database queries or resource-intensive operations. The steps include identifying and optimizing problematic queries, implementing indexing, limiting result sets, using connection pooling, scheduling maintenance, upgrading hardware or infrastructure, implementing caching, monitoring and analyzing performance, applying patches and updates, considering load balancing, and reviewing and optimizing application code. These steps should help resolve the CPU spike issue on prod-db-01 and prevent future occurrences.

Processing: Disk usage at 95% on prod-storage-02. Write latency > 800ms....
Agent conclusion:
The issue of high disk usage and increased write latency on `prod-storage-02` has been addressed with a step-by-step remediation plan. The root cause of insufficient di

### What's Now Agentic

- The LLM reads tool descriptions and decides which to call and in what order
- LOW severity alerts skip analyze_alert and fix_incident automatically, no hardcoded branching
- If the LLM is uncertain, it can loop — call analyze_alert again with different input, or escalate
- The full reasoning trace (every tool call and its output) is logged to Delta, not just the final answer
- We can add new tools (e.g.` query_logs`, `restart_service`) and the agent incorporates them without graph rewiring